In [7]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import skew
from tqdm import tqdm

DATA_ROOT = "data"
CLEAN_DIR = "data/in-hospital-mortality-cleaned"
LIST_DIR = "data/in-hospital-mortality"
OUTPUT_DIR = "data/in-hospital-mortality-7subseq-features"

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

import warnings
warnings.filterwarnings("ignore")



In [8]:
SUBSEQS = {
    "full": (0, 48),
    "first10": (0, 5),
    "first25": (0, 12),
    "first50": (0, 24),
    "last50": (24, 48),
    "last25": (36, 48),
    "last10": (43, 48),
}


In [9]:
def compute_stats(values):
    values = values.dropna()

    if len(values) == 0:
        return {
            "mean": np.nan,
            "std": np.nan,
            "min": np.nan,
            "max": np.nan,
            "skew": np.nan,
            "count": 0,
        }

    return {
        "mean": values.mean(),
        "std": values.std(),
        "min": values.min(),
        "max": values.max(),
        "skew": skew(values),
        "count": len(values),
    }


In [10]:
def extract_patient_features(file_path):
    df = pd.read_csv(file_path)
    df = df.sort_values("Hours")

    variables = [
        c for c in df.columns
        if c != "Hours"
        and not c.endswith("_missing")
        and not c.endswith("_time_since")
        and not c.endswith("_delta")
    ]

    features = {}

    for var in variables:
        series = pd.to_numeric(df[var], errors="coerce")

        for name, (start, end) in SUBSEQS.items():
            subseq = series.iloc[start:end]
            stats = compute_stats(subseq)

            # Statistical features
            for stat_name, val in stats.items():
                feature_name = f"{var}_{name}_{stat_name}"
                features[feature_name] = val

            # Missing indicator
            missing_flag = int(subseq.dropna().shape[0] == 0)
            features[f"{var}_{name}_missing"] = missing_flag

    return features


In [11]:
def process_split(split):
    listfile_path = os.path.join(LIST_DIR, f"{split}_listfile.csv")
    listfile = pd.read_csv(listfile_path)

    features = []

    for _, row in tqdm(listfile.iterrows(), total=len(listfile)):
        stay = row["stay"]
        label = row["y_true"]

        if split == "val":
            ts_path = os.path.join(CLEAN_DIR, "train", stay)
        else:
            ts_path = os.path.join(CLEAN_DIR, split, stay)

        feat = extract_patient_features(ts_path)
        feat["label"] = label
        features.append(feat)

    df = pd.DataFrame(features)
    out_path = os.path.join(OUTPUT_DIR, f"{split}_features.csv")
    df.to_csv(out_path, index=False)

    print(f"Saved {out_path}")


In [12]:
for split in ["train", "test"]:
    process_split(split)

print("Feature extraction complete.")


100%|██████████| 17903/17903 [07:17<00:00, 40.96it/s]


Saved data/in-hospital-mortality-7subseq-features/train_features.csv


100%|██████████| 3236/3236 [01:18<00:00, 40.99it/s]


Saved data/in-hospital-mortality-7subseq-features/test_features.csv
Feature extraction complete.
